# H4: модель CAMS против станции

**Что такое CAMS.** В `cams_pm25` записаны не натурные измерения датчика, а модельные значения атмосферного реанализа **Copernicus CAMS** (ячейка сетки `43.30° с.ш., 76.90° в.д.`, данные из Open-Meteo Air Quality API; первые непустые значения для Алматы начинаются с `2022-08-04`). Мы сравниваем эту модель с эталонной станцией **OpenAQ ID 8876** (Алматы, провайдер AirNow).

**Формулировки гипотез (до расчётов):**
- **H0:** в среднем CAMS совпадает со станцией (`bias = 0`, где `bias = среднее(CAMS − станция)`).
- **H1:** CAMS систематически отличается от станции (`bias ≠ 0`).

**Как принимаем решение:** оцениваем **95 % bootstrap доверительный интервал (CI)** суточного `bias` (`α = 0.05` ↔ 95 % CI). Если CI не содержит `0`, отвергаем H0. Дополнительно оцениваем размер эффекта (`bias` в µg/m³ и %), `MAE`, корреляции Пирсона `r` и Спирмена `ρ` и строим график сравнения. Гипотеза H4 проверяется по доверительному интервалу и является отдельной от подтверждающих p-тестов (`m = 5`) из `notebooks/03_eda.ipynb`.

**Почему главный уровень — сутки, а часы — дополнительно:** на часах `MAE` и корреляция сильно зависят от того, насколько точно совпадают суточные циклы станции и модели, которые сдвинуты друг относительно друга на несколько часов. Суточное среднее сглаживает внутрисуточный сдвиг на 1–3 ч, а `bias` (разница средних) к такому сдвигу почти нечувствителен, поэтому `bias` оцениваем на обоих уровнях, но основные выводы по H4 и согласию рядов делаем по суткам.

**Оговорки по времени (ADR-017):**
- Время CAMS — гарантированный **UTC**. Метки станции анализируются **при допущении, что они тоже в UTC**: версия сдвига `−6` ч не исключена, а сдвиг на `1–3` ч нашими проверками не различим.
- Сравниваем ряды строго на **лаге 0** по `time_utc` и ничего не сдвигаем во времени: лаговая проверка уже проведена в `notebooks/03_eda.ipynb` (часть 2, шаг 16, график 12). За весь период с учётом интерполированных часов (`n = 25 780`) на лаге `0` получено `ρ = 0.534`, на лучшем лаге `−2` ч — `ρ = 0.556`, на `+6` ч — `0.377`, а при 24-часовом сглаживании на лаге `0` — `0.770` (4.2, `03_eda`, шаг 16). Подбирать сдвиг под максимум корреляции на тех же данных означало бы подгонять результат задним числом, поэтому здесь лаги повторно не пересчитываем.

In [1]:
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sys.path.append("..")
from src import cams, clean, features, hypotheses

HOURLY_PATH = "../data/processed/hourly.csv"
DAILY_PATH = "../data/processed/daily.csv"
FIGURES_DIR = "../figures"

HEATING_LABELS = {True: "Отопительный сезон", False: "Вне сезона"}

# Единая тема графиков: палитра, удобная для дальтоников
sns.set_theme(style="whitegrid", context="talk", palette="colorblind")

## Шаг 1. Данные и константы

Загружаем очищенные таблицы `hourly.csv` и `daily.csv` из `data/processed/`, проверяем их структуру (`check_hourly`, `check_daily`) и задаём общие константы: число bootstrap-выборок, фиксированный `SEED` и шаг прореживания суток для проверки устойчивости к автокорреляции.

In [2]:
hourly = clean.read_clean_csv(HOURLY_PATH)
features.check_hourly(hourly)

daily = features.read_daily_csv(DAILY_PATH)
features.check_daily(daily)

# Параметры bootstrap (как в части 3 ноутбука 03_eda)
N_BOOT = 5000
SEED = 42
# Тот же шаг прореживания суток, что выбран в задаче 5.1 (03_eda, шаг 20)
K_THIN = 14

print(f"Строк в hourly: {len(hourly)} ({hourly['time_utc'].min()} … {hourly['time_utc'].max()})")
print(f"Строк в daily:  {len(daily)} ({daily['date'].min().date()} … {daily['date'].max().date()})")

Строк в hourly: 49075 (2020-04-09 16:00:00+00:00 … 2025-11-14 10:00:00+00:00)
Строк в daily:  2044 (2020-04-10 … 2025-11-13)


## Шаг 2. Часы, где есть оба значения

Для проверки H4 берём только реальные измеренные часы станции: строки, где заполнены и `pm25`, и `cams_pm25`, а флаг `pm25_interpolated == False`. Заполненные на этапе очистки короткие пропуски (до 3 часов) не являются натурными измерениями и в сравнение модели со станцией не входят.

Повторный `merge` на часовом уровне не нужен: оба ряда уже объединены в `hourly.csv` по ключу `time_utc` на этапе 3.1 (`validate="one_to_one"`). Дополнительно считаем коэффициент Спирмена на всех строках с обеими величинами (включая интерполированные), чтобы сверить данные с результатом из шага 16 `03_eda.ipynb`.

In [3]:
has_cams = hourly["cams_pm25"].notna()
both_with_interp = hourly["pm25"].notna() & has_cams
excluded_interp = both_with_interp & hourly["pm25_interpolated"].astype(bool)

print(f"Всего часов в hourly: {len(hourly)}")
print(f"Часов с непустым cams_pm25: {int(has_cams.sum())}")
print(f"Часов с обоими значениями (вкл. интерполированные): {int(both_with_interp.sum())}")
print(f"Исключено интерполированных часов станции: {int(excluded_interp.sum())}")

pairs = cams.paired_hours(hourly)
assert pairs["time_utc"].min() >= pd.Timestamp("2022-08-04", tz="UTC")

in_season = int(pairs["heating_season"].sum())
out_season = int((~pairs["heating_season"]).sum())
print(f"Итог парных измеренных часов (pairs): n = {len(pairs)}")
print(f"Период пар (UTC): {pairs['time_utc'].min()} … {pairs['time_utc'].max()}")
print(f"В отопительный сезон: {in_season} | Вне сезона: {out_season}")

# Сверка с шагом 16 (4.2, 03_eda): ρ Спирмена с интерполированными часами на лаге 0
check_with_interp = cams.agreement_metrics(hourly.loc[both_with_interp])
print(
    f"Сверка с 4.2 (с интерполированными): n = {check_with_interp['n']}, "
    f"spearman_rho = {check_with_interp['spearman_rho']:.6f}"
)

Всего часов в hourly: 49075
Часов с непустым cams_pm25: 28763
Часов с обоими значениями (вкл. интерполированные): 25780
Исключено интерполированных часов станции: 222
Итог парных измеренных часов (pairs): n = 25558
Период пар (UTC): 2022-09-06 18:00:00+00:00 … 2025-11-14 10:00:00+00:00
В отопительный сезон: 12191 | Вне сезона: 13367
Сверка с 4.2 (с интерполированными): n = 25780, spearman_rho = 0.533752


**Что видим.** (вывод шага 2) Из `49 075` часов таблицы `hourly` прогноз CAMS заполнен в `28 763` строках, а оба значения (включая интерполированные часы станции) есть в `25 780` часах. После исключения `222` интерполированных часов для проверки H4 остаётся `25 558` парных измеренных часов за период `2022-09-06 18:00 UTC … 2025-11-14 10:00 UTC`: `12 191` час в отопительный сезон и `13 367` часов вне сезона. Контрольный расчёт по всем `25 780` часам с интерполяцией даёт `spearman_rho = 0.533752`, что совпадает со значением из шага 16 `03_eda.ipynb` (`0.534`, `n = 25 780`).

## Шаг 3. CAMS и станция по часам

Оцениваем согласие модели CAMS со станцией на парных измеренных часах за весь период, в отопительный сезон и вне его.

**Метрики:**
- `bias = среднее(CAMS − станция)` в µg/m³ (и `bias_pct` — доля от среднего станции в %). Отрицательный знак означает, что CAMS в среднем занижает концентрацию PM2.5, положительный — завышает.
- `MAE = среднее(|CAMS − станция|)` в µg/m³ — средняя абсолютная ошибка без взаимного погашения знаков.
- Корреляции **Пирсона `r`** (линейная связь, как в постановке задачи) и **Спирмена `ρ`** (ранговая связь, устойчивая к редким пиковым выбросам PM2.5).

Доверительные интервалы на часовом уровне не строим: соседние часы сильно похожи друг на друга, поэтому bootstrap по ~25 тыс. часов дал бы обманчиво узкий интервал. Кроме того, почасовые `MAE` и `ρ` чувствительны к совмещению суточных циклов станции и модели (ADR-017: метки станции анализируем при допущении UTC, возможен сдвиг фазы), поэтому часовой уровень считаем дополнительным, а основные выводы делаем по суткам в шаге 4.

In [4]:
GROUPS = {
    "весь период": pd.Series(True, index=pairs.index),
    "отопительный сезон": pairs["heating_season"],
    "вне сезона": ~pairs["heating_season"],
}

hourly_rows = []
for group_name, mask in GROUPS.items():
    metrics = cams.agreement_metrics(pairs.loc[mask])
    hourly_rows.append({"grain": "часы", "group": group_name, **metrics})

hourly_table = pd.DataFrame(hourly_rows)
print(hourly_table.round(3).to_string(index=False))

grain              group     n  station_mean  cams_mean    bias  bias_pct    mae  pearson_r  spearman_rho
 часы        весь период 25558        26.435     21.971  -4.464   -16.886 15.878      0.504         0.533
 часы отопительный сезон 12191        41.651     28.635 -13.015   -31.249 25.845      0.402         0.442
 часы         вне сезона 13367        12.557     15.892   3.335    26.562  6.788      0.276         0.341


**Что видим.** (вывод шага 3)
- **Весь период** (`n = 25 558`): среднее станции `26.435` µg/m³, CAMS `21.971` µg/m³; `bias = -4.464` µg/m³ (`-16.886 %`), `MAE = 15.878` µg/m³, `r = 0.504`, `ρ = 0.533`. Часовая ранговая корреляция `ρ = 0.533` практически совпадает со значением из 4.2 (`0.534` на `25 780` часах с интерполированными, сверено в шаге 2): исключение интерполированных часов почти не меняет оценку связи.
- **Отопительный сезон** (`n = 12 191`): среднее станции `41.651` µg/m³, CAMS `28.635` µg/m³; CAMS занижает на `bias = -13.015` µg/m³ (`-31.249 %`), `MAE = 25.845` µg/m³, `r = 0.402`, `ρ = 0.442`.
- **Вне сезона** (`n = 13 367`): среднее станции `12.557` µg/m³, CAMS `15.892` µg/m³; знак смещения меняется — CAMS завышает на `bias = 3.335` µg/m³ (`26.562 %`), `MAE = 6.788` µg/m³, `r = 0.276`, `ρ = 0.341`.

При противоположных знаках по сезонам (`-13.015` µg/m³ в отопительный сезон и `+3.335` µg/m³ вне его) общий `bias` (`-4.464` µg/m³) частично сглаживается, тогда как `MAE` за весь период достигает `15.878` µg/m³.

## Шаг 4. CAMS и станция по суткам

Переходим к главному уровню проверки H4 — суткам. Парные измеренные часы группируем по местной дате (`Asia/Almaty`) и оставляем средние только для суток с не менее чем 18 парными часами (`MIN_HOURS_PER_DAY`). Так станция и CAMS усредняются строго по одним и тем же часам (из-за этого суточное `pm25` здесь может немного отличаться от `pm25_mean` в `daily.csv`, где порог 18 часов проверялся независимо от наличия CAMS). Затем присоединяем суточные пары к полному календарю `daily.csv` через `merge` (`validate="one_to_one"`), чтобы сохранить непрерывный ряд дат для проверки автокорреляции в шаге 5.

Для суточных `bias` и `MAE` строим 95 % перцентильные bootstrap-интервалы (лекция 8, `N_BOOT = 5000`). Используем один и тот же `SEED` для обеих метрик: при одинаковом размере группы индексы выборок с возвращением совпадают, поэтому `bias` и `MAE` оцениваются на одних и тех же bootstrap-наборах суток. Для ранговой корреляции `ρ` за весь период дополнительно считаем 95 % bootstrap CI парами суток (`spearman_with_ci`).

**Критерий решения по H4:** если 95 % bootstrap CI суточного `bias` за весь период не содержит `0`, отвергаем H0 при `α = 0.05`.

In [5]:
daily_table = cams.daily_pairs(pairs, daily)
valid_days = daily_table.loc[daily_table["diff"].notna()]
in_season_days = int(valid_days["heating_season"].sum())
out_season_days = int((~valid_days["heating_season"]).sum())

print(f"Строк в daily_table (по календарю daily): {len(daily_table)}")
print(f"Суток с парными средними (>= {features.MIN_HOURS_PER_DAY} ч): {len(valid_days)}")
print(f"Период суток: {valid_days['date'].min().date()} … {valid_days['date'].max().date()}")
print(f"В отопительный сезон: {in_season_days} | Вне сезона: {out_season_days}")

daily_groups = {
    "весь период": pd.Series(True, index=daily_table.index),
    "отопительный сезон": daily_table["heating_season"],
    "вне сезона": ~daily_table["heating_season"],
}

daily_rows = []
for group_name, mask in daily_groups.items():
    sub = daily_table.loc[mask & daily_table["diff"].notna()]
    metrics = cams.agreement_metrics(sub)
    # Один и тот же SEED: на одинаковой длине sub индексные выборки для bias и MAE совпадают
    bias_ci_low, bias_ci_high = hypotheses.percentile_ci(
        cams.bootstrap_means(sub["diff"], N_BOOT, SEED)
    )
    mae_ci_low, mae_ci_high = hypotheses.percentile_ci(
        cams.bootstrap_means(sub["diff"].abs(), N_BOOT, SEED)
    )
    daily_rows.append(
        {
            "grain": "сутки",
            "group": group_name,
            **metrics,
            "bias_ci_low": bias_ci_low,
            "bias_ci_high": bias_ci_high,
            "mae_ci_low": mae_ci_low,
            "mae_ci_high": mae_ci_high,
        }
    )

daily_metrics = pd.DataFrame(daily_rows)
print("\nМетрики согласия на суточном уровне (с 95 % bootstrap CI):")
print(daily_metrics.round(3).to_string(index=False))

daily_rho = hypotheses.spearman_with_ci(daily_table, "pm25", "cams_pm25", N_BOOT, SEED)
print(
    f"\nСпирмен по суткам (весь период): n = {daily_rho['n']}, "
    f"ρ = {daily_rho['rho']:.3f}, "
    f"95 % CI ({daily_rho['ci_low']:.3f} … {daily_rho['ci_high']:.3f}), "
    f"p = {hypotheses.format_p(daily_rho['p'])} ({daily_rho['p']:.3e})"
)

overall = daily_metrics.loc[daily_metrics["group"] == "весь период"].iloc[0]
h4_reject = not (overall["bias_ci_low"] <= 0 <= overall["bias_ci_high"])
decision_text = "H0 отвергаем" if h4_reject else "H0 не отвергаем"
print(
    f"Решение по H4 (95 % CI суточного bias не содержит 0): "
    f"[{overall['bias_ci_low']:.3f}, {overall['bias_ci_high']:.3f}] → {decision_text}"
)

Строк в daily_table (по календарю daily): 2044
Суток с парными средними (>= 18 ч): 1057
Период суток: 2022-09-07 … 2025-11-13
В отопительный сезон: 502 | Вне сезона: 555



Метрики согласия на суточном уровне (с 95 % bootstrap CI):
grain              group    n  station_mean  cams_mean    bias  bias_pct    mae  pearson_r  spearman_rho  bias_ci_low  bias_ci_high  mae_ci_low  mae_ci_high
сутки        весь период 1057        25.997     21.805  -4.192   -16.125 11.243      0.652         0.767       -5.454        -2.984      10.189       12.346
сутки отопительный сезон  502        40.878     28.387 -12.491   -30.557 18.602      0.518         0.626      -14.856       -10.144      16.657       20.624
сутки         вне сезона  555        12.537     15.851   3.314    26.437  4.588      0.481         0.540        2.889         3.755       4.255        4.943



Спирмен по суткам (весь период): n = 1057, ρ = 0.767, 95 % CI (0.737 … 0.794), p = < 0.001 (1.245e-205)
Решение по H4 (95 % CI суточного bias не содержит 0): [-5.454, -2.984] → H0 отвергаем


**Что видим.** (вывод шага 4) В календарную таблицу `daily_table` вошли все `2044` суток `daily.csv`, из них парные средние (не менее `18` измеренных часов) есть для `1057` суток за период `2022-09-07 … 2025-11-13` (`502` в отопительный сезон и `555` вне сезона).

- **Весь период** (`n = 1057`): среднее станции `25.997` µg/m³, CAMS `21.805` µg/m³; `bias = -4.192` µg/m³ (`-16.125 %`), 95 % CI `[-5.454, -2.984]` µg/m³; `MAE = 11.243` µg/m³, 95 % CI `[10.189, 12.346]` µg/m³; корреляции `r = 0.652` и `ρ = 0.767` с 95 % CI `(0.737 … 0.794)` (`p < 0.001`).
- **Отопительный сезон** (`n = 502`): станция `40.878` µg/m³, CAMS `28.387` µg/m³; CAMS заметно занижает — `bias = -12.491` µg/m³ (`-30.557 %`), 95 % CI `[-14.856, -10.144]` µg/m³; `MAE = 18.602` µg/m³ (`[16.657, 20.624]`); `r = 0.518`, `ρ = 0.626`.
- **Вне сезона** (`n = 555`): станция `12.537` µg/m³, CAMS `15.851` µg/m³; знак смещения меняется на противоположный — CAMS завышает на `bias = 3.314` µg/m³ (`26.437 %`), 95 % CI `[2.889, 3.755]` µg/m³; `MAE = 4.588` µg/m³ (`[4.255, 4.943]`); `r = 0.481`, `ρ = 0.540`.

**Различие по сезонам и решение по H4:** суточное усреднение заметно повышает согласие колебаний по сравнению с часами (`ρ = 0.767` против `0.533`), однако знак смещения по сезонам противоположен (`-12.491` µg/m³ в отопительный сезон и `+3.314` µg/m³ вне сезона). Для всего периода 95 % bootstrap CI суточного `bias` `[-5.454, -2.984]` µg/m³ не содержит `0` — **H0 отвергаем**: в среднем CAMS систематически отличается от измерений станции (занижает на `4.192` µg/m³, или `16.125 %`). Устойчивость этого интервала к зависимости соседних суток проверим в шаге 5.

## Проверка: часы с почти нулевым CAMS

В сырых данных `data/raw/cams_aq.csv` начиная с `2022-12-13 12:00 UTC` значение CAMS на несколько суток падает до долей µg/m³, хотя станция в эти часы показывает высокие значения. Это похоже на сбой модельного ряда. Считаем, сколько парных часов с `cams_pm25 < 1` и в какие даты они лежат, затем пересчитываем суточные метрики без этих часов (весь период и отопительный сезон), чтобы понять, зависит ли от них вывод H4.

In [6]:
# Порог: значения CAMS ниже 1 µg/m³ при высокой станции считаем подозрительными
LOW_CAMS = 1.0

low_hours = pairs.loc[pairs["cams_pm25"] < LOW_CAMS]
print(f"Парных часов с cams_pm25 < {LOW_CAMS}: {len(low_hours)}")
print(f"Станция в эти часы: от {low_hours['pm25'].min():.1f} до {low_hours['pm25'].max():.1f} µg/m³")

# Начало и конец эпизода и проверка, что часы идут подряд без пропусков
episode_start = low_hours["time_utc"].min()
episode_end = low_hours["time_utc"].max()
is_continuous = (low_hours["time_utc"].diff().dropna() == pd.Timedelta("1h")).all()
print(f"Первый час: {episode_start}, последний час: {episode_end}")
print(f"Часы идут подряд без пропусков: {is_continuous}")

# Даты (UTC) и число таких часов в каждой дате
low_by_date = low_hours["time_utc"].dt.date.value_counts().sort_index()
print("Даты (UTC) и число часов:")
print(low_by_date.to_string())

# Те же сутки-метрики без этих часов
pairs_clean = pairs.loc[pairs["cams_pm25"] >= LOW_CAMS]
daily_clean = cams.daily_pairs(pairs_clean, daily)
valid_clean = daily_clean.loc[daily_clean["diff"].notna()]
print(f"\nСуток с парными средними без этих часов: {len(valid_clean)} (было {len(valid_days)})")

sens_rows = []
for group_name, base_mask in {
    "весь период": pd.Series(True, index=daily_clean.index),
    "отопительный сезон": daily_clean["heating_season"],
}.items():
    sub = valid_clean.loc[base_mask.loc[valid_clean.index]]
    metrics = cams.agreement_metrics(sub)
    ci_low, ci_high = hypotheses.percentile_ci(cams.bootstrap_means(sub["diff"], N_BOOT, SEED))
    sens_rows.append(
        {
            "group": group_name,
            "n": metrics["n"],
            "bias": metrics["bias"],
            "bias_ci_low": ci_low,
            "bias_ci_high": ci_high,
            "mae": metrics["mae"],
            "spearman_rho": metrics["spearman_rho"],
        }
    )

sens_table = pd.DataFrame(sens_rows)
print("\nСуточные метрики без часов с низким CAMS:")
print(sens_table.round(3).to_string(index=False))

sens_all = sens_table.loc[sens_table["group"] == "весь период"].iloc[0]
sens_reject = not (sens_all["bias_ci_low"] <= 0 <= sens_all["bias_ci_high"])
print(f"Решение по H4 без этих часов: {'H0 отвергаем' if sens_reject else 'H0 не отвергаем'}")

Парных часов с cams_pm25 < 1.0: 59
Станция в эти часы: от 10.0 до 264.0 µg/m³
Первый час: 2022-12-13 12:00:00+00:00, последний час: 2022-12-15 22:00:00+00:00
Часы идут подряд без пропусков: True
Даты (UTC) и число часов:
time_utc
2022-12-13    12
2022-12-14    24
2022-12-15    23

Суток с парными средними без этих часов: 1055 (было 1057)

Суточные метрики без часов с низким CAMS:
             group    n    bias  bias_ci_low  bias_ci_high    mae  spearman_rho
       весь период 1055  -3.925       -5.117        -2.750 10.989         0.777
отопительный сезон  500 -11.960      -14.225        -9.718 18.095         0.645
Решение по H4 без этих часов: H0 отвергаем


**Что видим.** (вывод проверки низких значений CAMS) Среди `25 558` парных часов `59` часов имеют `cams_pm25 < 1` µg/m³: `12` часов `2022-12-13`, `24` часа `2022-12-14` и `23` часа `2022-12-15` (UTC), то есть один непрерывный эпизод (проверка непрерывности: `True`) с `2022-12-13 12:00 UTC` по `2022-12-15 22:00 UTC`, около трёх суток; станция в эти часы показывает от `10.0` до `264.0` µg/m³. Такой провал модельного ряда при высоких значениях станции похож на сбой CAMS, а не на реальную чистоту воздуха. Без этих часов остаётся `1055` суток вместо `1057`.

| Период | Было (`n`, `bias`, `MAE`, `ρ`) | Без низких часов (`n`, `bias`, 95 % CI, `MAE`, `ρ`) |
|---|---|---|
| Весь период | `1057`, `-4.192`, `11.243`, `0.767` | `1055`, `-3.925`, `[-5.117, -2.750]`, `10.989`, `0.777` |
| Отопительный сезон | `502`, `-12.491`, `18.602`, `0.626` | `500`, `-11.960`, `[-14.225, -9.718]`, `18.095`, `0.645` |

Эти часы немного усиливают занижение (`bias` без них на `0.267` µg/m³ слабее по модулю), но интервал суточного `bias` без них `[-5.117, -2.750]` µg/m³ по-прежнему не содержит `0`: вывод H4 (**H0 отвергаем**) от них не зависит. Основные числа выше оставляем по всем парным часам (решение о фильтре принято по результату проверки, а не до неё).

## Шаг 5. Держится ли вывод, если соседние сутки похожи

Обычный bootstrap (лекция 8) предполагает независимость наблюдений (IID). Для суточной разницы `diff = cams_pm25 − pm25` соседние сутки похожи друг на друга, из-за чего доверительный интервал по всем суткам подряд может оказаться узковатым.

Сначала измеряем автокорреляцию Спирмена `ρ` для разницы `diff` на лагах `1`, `7` и `14` суток (`hypotheses.autocorrelation_by_lag`). Затем проверяем устойчивость вывода H4 с помощью **прореживания календарных суток** (`hypotheses.thin_days`): берём каждые `K_THIN = 14` календарные сутки (шаг `k = 14` уже выбран и объяснён в шаге 20 `notebooks/03_eda.ipynb`) и для каждого из `14` сдвигов начала (`offset = 0…13`) пересчитываем средний `bias` и его 95 % bootstrap CI.

**Критерий устойчивости (до результатов):** считаем вывод H4 полностью устойчивым к зависимости соседних суток, если 95 % CI `bias` не содержит `0` во всех `14` сдвигах.

In [7]:
autocorr = hypotheses.autocorrelation_by_lag(daily_table, "diff", [1, 7, 14])
print("Автокорреляция Спирмена разницы diff (cams_pm25 − pm25):")
print(autocorr.round({"rho": 3}).to_string(index=False))

thin_rows = []
for offset in range(K_THIN):
    sub = hypotheses.thin_days(daily_table, K_THIN, offset)
    sub = sub[sub["diff"].notna()]
    bias = float(sub["diff"].mean())
    ci_low, ci_high = hypotheses.percentile_ci(
        cams.bootstrap_means(sub["diff"], N_BOOT, SEED)
    )
    thin_rows.append(
        {
            "offset": offset,
            "n": int(len(sub)),
            "bias": bias,
            "ci_low": ci_low,
            "ci_high": ci_high,
            "excludes_zero": bool(not (ci_low <= 0 <= ci_high)),
        }
    )

thin_table = pd.DataFrame(thin_rows)
assert len(thin_table) == K_THIN

print(f"\nПрореживание суток (K_THIN = {K_THIN}, все {K_THIN} сдвигов начала):")
print(thin_table.round({"bias": 3, "ci_low": 3, "ci_high": 3}).to_string(index=False))

n_excludes = int(thin_table["excludes_zero"].sum())
print(f"\nДиапазон n: {thin_table['n'].min()}–{thin_table['n'].max()} суток")
print(
    f"Диапазон bias: {thin_table['bias'].min():.3f} … {thin_table['bias'].max():.3f} µg/m³"
)
print(f"CI не содержит 0 в {n_excludes} из {K_THIN} сдвигов")

Автокорреляция Спирмена разницы diff (cams_pm25 − pm25):
 lag   rho    n
   1 0.458 1032
   7 0.302  993
  14 0.208  976



Прореживание суток (K_THIN = 14, все 14 сдвигов начала):
 offset  n   bias  ci_low  ci_high  excludes_zero
      0 75 -4.505  -8.911   -0.773           True
      1 75 -5.319 -11.353   -0.346           True
      2 75 -4.478  -9.665   -0.200           True
      3 74 -4.156  -9.739    0.695          False
      4 75 -2.546  -6.828    1.200          False
      5 74 -2.137  -7.430    2.159          False
      6 73 -1.535  -6.231    2.505          False
      7 75 -2.767  -7.290    1.155          False
      8 76 -3.103  -6.430   -0.198           True
      9 76 -4.245  -8.004   -0.703           True
     10 77 -5.587 -11.093   -1.011           True
     11 77 -4.701  -8.984   -1.008           True
     12 79 -7.840 -13.416   -3.042           True
     13 76 -5.423 -11.778   -0.153           True

Диапазон n: 73–79 суток
Диапазон bias: -7.840 … -1.535 µg/m³
CI не содержит 0 в 9 из 14 сдвигов


**Что видим.** (вывод шага 5)
- Автокорреляция Спирмена разницы `diff` на лаге `1` сутки равна `0.458` (`n = 1032`), на лаге `7` суток — `0.302` (`n = 993`), на лаге `14` суток снижается до `0.208` (`n = 976`): зависимость соседних суток умеренная и постепенно затухает к двум неделям.
- После прореживания с шагом `K_THIN = 14` в каждом из `14` сдвигов (`offset = 0…13`) остаётся `73–79` суток (вместо `1057` в шаге 4).
- Средний `bias` остаётся отрицательным во всех `14` сдвигах — в диапазоне `-7.840 … -1.535` µg/m³ (самый сильный сдвиг на `offset = 12`: `-7.840` µg/m³, 95 % CI `[-13.416, -3.042]`; самый слабый на `offset = 6`: `-1.535` µg/m³, 95 % CI `[-6.231, 2.505]`).
- 95 % CI `bias` не содержит `0` в `9` из `14` сдвигов (`offset` `0, 1, 2` и `8–13`), а в `5` сдвигах (`3–7`) ноль попадает в интервал.
- **Устойчивость вывода по H4:** по строгому критерию (отсутствие нуля в CI во всех `14` сдвигах) отклонение H0 выдерживает проверку лишь частично (`9` из `14` сдвигов), хотя направление смещения (`bias < 0`) сохраняется в каждом сдвиге. Поскольку прореживание сокращает выборку до `73–79` суток, а полный интервал `[-5.454, -2.984]` µg/m³ (шаг 4) рассчитан без поправки на автокорреляцию (`ρ = 0.458` на лаге `1`), фиксируем зависимость соседних суток как ограничение оценки доверительного интервала.

## Шаг 6. График: CAMS против станции

Строим диаграмму рассеяния модели CAMS против измерений станции на двух уровнях: слева — все парные измеренные часы, справа — суточные средние с разделением на отопительный сезон и время вне сезона.

**Как читать линию `y = x`:** пунктирная диагональ показывает точное совпадение модели и станции (`CAMS = станция`). Точки **ниже** линии означают, что CAMS занижает PM2.5, точки **выше** линии — завышает. Чтобы диагональ не искажала наклон на глаз, на каждой панели задаём одинаковые пределы осей X и Y от `0` до общего максимума обеих осей.

In [8]:
h_all = hourly_table.loc[hourly_table["group"] == "весь период"].iloc[0]
d_all = daily_metrics.loc[daily_metrics["group"] == "весь период"].iloc[0]
d_heat = daily_metrics.loc[daily_metrics["group"] == "отопительный сезон"].iloc[0]
d_off = daily_metrics.loc[daily_metrics["group"] == "вне сезона"].iloc[0]

hour_max = float(max(pairs["pm25"].max(), pairs["cams_pm25"].max()))
day_max = float(max(valid_days["pm25"].max(), valid_days["cams_pm25"].max()))

palette = sns.color_palette()
fig, (ax_hour, ax_day) = plt.subplots(1, 2, figsize=(16, 7))

# Левая панель: все парные измеренные часы
ax_hour.scatter(pairs["pm25"], pairs["cams_pm25"], s=4, alpha=0.1, color=palette[0])
ax_hour.plot([0, hour_max], [0, hour_max], color="black", linestyle="--", label="CAMS = станция")
ax_hour.set_xlim(0, hour_max)
ax_hour.set_ylim(0, hour_max)
ax_hour.set_aspect("equal", adjustable="box")
ax_hour.set_xlabel("Станция, PM2.5 µg/m³")
ax_hour.set_ylabel("CAMS, PM2.5 µg/m³")
ax_hour.set_title(
    f"Часы (n = {int(h_all['n'])}, bias = {h_all['bias']:.2f} µg/m³, ρ = {h_all['spearman_rho']:.3f})"
)
ax_hour.legend(loc="upper left")

# Правая панель: суточные средние по сезонам
for flag in [True, False]:
    part = valid_days.loc[valid_days["heating_season"] == flag]
    ax_day.scatter(
        part["pm25"],
        part["cams_pm25"],
        s=18,
        alpha=0.55,
        label=HEATING_LABELS[flag],
    )
ax_day.plot([0, day_max], [0, day_max], color="black", linestyle="--", label="CAMS = станция")
ax_day.set_xlim(0, day_max)
ax_day.set_ylim(0, day_max)
ax_day.set_aspect("equal", adjustable="box")
ax_day.set_xlabel("Станция, PM2.5 µg/m³")
ax_day.set_ylabel("CAMS, PM2.5 µg/m³")
ax_day.set_title(
    f"Сутки (n = {int(d_all['n'])}, bias = {d_all['bias']:.2f} µg/m³, ρ = {daily_rho['rho']:.3f})"
)
ax_day.legend(loc="upper left")

# Слово направления смещения выбирает код по знаку bias
bias_day = float(d_all["bias"])
bias_pct_day = float(d_all["bias_pct"])
bias_heat = float(d_heat["bias"])
rho_day = float(daily_rho["rho"])
direction_all = "занижает" if bias_day < 0 else "завышает"
direction_heat = "занижает" if bias_heat < 0 else "завышает"

title_text = (
    f"CAMS {direction_all} суточный PM2.5 на {abs(bias_day):.1f} µg/m³ ({bias_pct_day:+.0f} %), "
    f"в отопительный сезон {direction_heat} на {abs(bias_heat):.1f} µg/m³; "
    f"ρ по суткам = {rho_day:.2f}"
)
fig.suptitle(title_text, y=1.01)
fig.tight_layout()
fig.savefig(f"{FIGURES_DIR}/08_cams_vs_station.png", dpi=150, bbox_inches="tight")
plt.close(fig)

print(f"График 8 сохранён: {FIGURES_DIR}/08_cams_vs_station.png")
print(f"Заголовок: {title_text}")
print(f"Пределы осей (0..max): часы = {hour_max:.2f} µg/m³, сутки = {day_max:.2f} µg/m³")
print(
    f"Сутки по сезонам: отопительный bias = {bias_heat:.2f} µg/m³, "
    f"вне сезона bias = {float(d_off['bias']):.2f} µg/m³"
)

График 8 сохранён: ../figures/08_cams_vs_station.png
Заголовок: CAMS занижает суточный PM2.5 на 4.2 µg/m³ (-16 %), в отопительный сезон занижает на 12.5 µg/m³; ρ по суткам = 0.77
Пределы осей (0..max): часы = 375.00 µg/m³, сутки = 182.29 µg/m³
Сутки по сезонам: отопительный bias = -12.49 µg/m³, вне сезона bias = 3.31 µg/m³


**Что видим.** (вывод шага 6) На часовой панели (`n = 25 558`, пределы `0…375.00` µg/m³, `bias = -4.46` µg/m³, `ρ = 0.533`) и на суточной панели (`n = 1057`, пределы `0…182.29` µg/m³, `bias = -4.19` µg/m³, `ρ = 0.767`) при повышенном загрязнении облако точек заметно уходит ниже диагонали `y = x`: при высоких значениях станции модель CAMS остаётся в более узком диапазоне.

На правой панели отчётливо видны два режима: в **отопительный сезон** точки в основном лежат ниже линии `y = x` (`bias = -12.49` µg/m³), а **вне сезона** при низких концентрациях основная часть точек смещена немного выше диагонали (`bias = +3.31` µg/m³).

Точки у самой оси `CAMS ≈ 0` при высокой станции — это `59` часов `2022-12-13 … 2022-12-15` с `cams_pm25 < 1` (проверка выше): вероятный сбой модельного ряда, а не реальная чистота воздуха. Без этих часов суточный `bias = -3.925` µg/m³ (`95 % CI [-5.117, -2.750]`), то есть вывод H4 от них не зависит.

![График 8](../figures/08_cams_vs_station.png)

## Шаг 7. Итоговая таблица H4

Собираем часовые и суточные метрики согласия в единую сводную таблицу `h4_table` (`6` строк: `2` уровня агрегации × `3` периода).

In [9]:
h4_table = pd.concat([hourly_table, daily_metrics], ignore_index=True)
assert len(h4_table) == 6, "в итоговой таблице H4 должно быть 6 строк"

print("Итоговая таблица H4 (часы и сутки):")
print(h4_table.round(3).to_string(index=False))

Итоговая таблица H4 (часы и сутки):
grain              group     n  station_mean  cams_mean    bias  bias_pct    mae  pearson_r  spearman_rho  bias_ci_low  bias_ci_high  mae_ci_low  mae_ci_high
 часы        весь период 25558        26.435     21.971  -4.464   -16.886 15.878      0.504         0.533          NaN           NaN         NaN          NaN
 часы отопительный сезон 12191        41.651     28.635 -13.015   -31.249 25.845      0.402         0.442          NaN           NaN         NaN          NaN
 часы         вне сезона 13367        12.557     15.892   3.335    26.562  6.788      0.276         0.341          NaN           NaN         NaN          NaN
сутки        весь период  1057        25.997     21.805  -4.192   -16.125 11.243      0.652         0.767       -5.454        -2.984      10.189       12.346
сутки отопительный сезон   502        40.878     28.387 -12.491   -30.557 18.602      0.518         0.626      -14.856       -10.144      16.657       20.624
сутки         вн

## Итог H4

1. **Проверка H4 и размер эффекта.** На общем периоде с измерениями станции и прогнозом CAMS (`25 558` парных измеренных часов за `2022-09-06 18:00 UTC … 2025-11-14 10:00 UTC`, без `222` интерполированных часов; `1057` суток с `>= 18` парными часами за `2022-09-07 … 2025-11-13`) (вывод шага 2, вывод шага 4) средний суточный уровень на станции равен `25.997` µg/m³, у CAMS — `21.805` µg/m³ (вывод шага 4, вывод шага 7). Суточный `bias = -4.192` µg/m³ (`-16.125 %`), 95 % bootstrap CI `[-5.454, -2.984]` µg/m³ не содержит `0` → **H0 отвергаем**: в среднем CAMS систематически занижает концентрацию PM2.5 (вывод шага 4). Оговорка: это отклонение H0 опирается на интервал без поправки на зависимость соседних суток, а по заранее заданному критерию прореживания устойчиво лишь частично (`9` из `14` сдвигов), хотя знак `bias` отрицателен во всех `14` (вывод шага 5). Общий `bias` — смесь сезонов с разными знаками (`-12.491` и `+3.314` µg/m³) и зависит от доли сезонов (`502` и `555` суток), поэтому содержательный вывод — посезонный (пункт 2). На часовом уровне общий `bias` близок — `-4.464` µg/m³ (`-16.886 %`, `MAE = 15.878` µg/m³) (вывод шага 3, вывод шага 7). Часы с `cams_pm25 < 1` (`59` часов, `2022-12-13 … 2022-12-15`) на вывод не влияют: без них суточный `bias = -3.925` µg/m³, CI `[-5.117, -2.750]` µg/m³ (вывод проверки низких значений CAMS).
2. **Сезонное различие (где отличие больше).** Знак смещения по сезонам противоположен (вывод шага 4, вывод шага 6, вывод шага 7). В **отопительный сезон** (`n = 502` суток) CAMS занижает сильнее всего: суточный `bias = -12.491` µg/m³ (`-30.557 %`, 95 % CI `[-14.856, -10.144]` µg/m³), `MAE = 18.602` µg/m³ (`[16.657, 20.624]` µg/m³), на часах `bias = -13.015` µg/m³ (`-31.249 %`, `MAE = 25.845` µg/m³). **Вне сезона** (`n = 555` суток) CAMS, напротив, завышает: суточный `bias = +3.314` µg/m³ (`+26.437 %`, 95 % CI `[2.889, 3.755]` µg/m³), `MAE = 4.588` µg/m³ (`[4.255, 4.943]` µg/m³), на часах `bias = +3.335` µg/m³ (`+26.562 %`, `MAE = 6.788` µg/m³). Противоположные знаки по сезонам сглаживают общий `bias` (`-4.192` µg/m³), но средняя абсолютная ошибка по суткам за весь период составляет `11.243` µg/m³ (`[10.189, 12.346]` µg/m³).
3. **Согласие колебаний (`ρ` и `r`).** На главном суточном уровне ранговая корреляция Спирмена за весь период достигает `ρ = 0.767` (95 % CI `0.737 … 0.794`, `p < 0.001`), Пирсона `r = 0.652` (вывод шага 4, вывод шага 7). Внутри отдельных периодов корреляция умеренная — `ρ = 0.626` (`r = 0.518`) в отопительный сезон и `ρ = 0.540` (`r = 0.481`) вне сезона, то есть часть высокой общей корреляции отражает совместный сезонный ход между тёплым и холодным временем года. На часах согласие ниже (`ρ = 0.533`, `r = 0.504` за весь период) (вывод шага 3, вывод шага 7). Это значит, что CAMS воспроизводит направление суточных подъёмов и спадов PM2.5, но по уровню заметно расходится со станцией, поэтому без отдельной проверки заменить им измерения нельзя (калибровку мы не проверяли).
4. **Устойчивость к зависимости соседних суток.** Разница `diff = cams_pm25 − pm25` имеет автокорреляцию Спирмена `ρ = 0.458` на лаге `1` сутки, `0.302` на лаге `7` и `0.208` на лаге `14` (вывод шага 5). После прореживания календарных суток с шагом `K_THIN = 14` (`73–79` суток в подвыборке) средний `bias` остаётся отрицательным во всех `14` сдвигах (`-7.840 … -1.535` µg/m³), а 95 % CI не содержит `0` в `9` из `14` сдвигов (вывод шага 5). Словами: направление занижения устойчиво, но статистическая значимость при учёте зависимости суток подтверждается только частично — для `5` сдвигов (`3–7`) ноль попадает в интервал.

**Ограничения:**
- **Грубая пространственная ячейка CAMS:** модель даёт среднее по ячейке сетки размером в десятки километров, а не точечное измерение датчика; возможные физические причины расхождения (локальные источники, рельеф города, сглаживание пиков по ячейке) здесь не проверялись.
- **Одна опорная станция:** сравнение выполнено только по одной станции OpenAQ (`ID 8876`), поэтому разницу нельзя автоматически переносить на другие районы Алматы.
- **Метки времени станции (ADR-017):** время CAMS — точный UTC, а метки станции сравниваются на лаге `0` при допущении UTC (возможный сдвиг фазы на `1–3` ч не различим, версия `−6` ч не исключена). Так как внутрисуточные циклы чувствительны к совмещению фазы, часовые `MAE = 15.878` µg/m³ и `ρ = 0.533` зависят от этого допущения; основным уровнем выбраны сутки.
- **Зависимость соседних суток:** при автокорреляции разницы `ρ = 0.458` на лаге `1` стандартный bootstrap по всем суткам занижает ширину доверительного интервала, а прореживание до `73–79` суток снижает мощность проверки (`9` из `14` сдвигов без нуля в CI).
- **Вероятный сбой модельного ряда CAMS:** `59` часов `2022-12-13 … 2022-12-15` (UTC) с `cams_pm25 < 1` µg/m³ при станции до `264.0` µg/m³ — вероятно, сбой модели; в основные числа они включены, но без них вывод H4 прежний (суточный `bias = -3.925` µg/m³, CI `[-5.117, -2.750]` µg/m³).
- **Статус в семействе гипотез:** H4 проверяется по 95 % доверительному интервалу суточного `bias` и не входит в `m = 5` подтверждающих p-тестов из задачи 5.1 (`notebooks/03_eda.ipynb`).